# Lab 02: Primes, Primality Testing, and Special Sequences
**Student Name:** Md. Akib Al Jawad  
**Student ID:** IT-23020  
**Department:** Information and Communication Technology (ICT)  
**Institution:** Mawlana Bhashani Science and Technology University (MBSTU)  
**Reference:** IIT Bombay Maths and Physics Club — Number Theory Workshop (Day 2 Assignment & Slides)

---

## Overview
This notebook investigates three advanced theoretical problems in number theory:
1. **Repunit Sequences & Prime Divisibility**: Analysis of $a(n) = \underbrace{11\dots1}_{n \text{ ones}} = \frac{10^n - 1}{9}$. Proof that $A_2 = \emptyset$ and $A_5 = \emptyset$, while $A_p$ is infinite for all other primes.
2. **Euclid-Euler Theorem on Even Perfect Numbers**: Complete proof that an even integer $k$ is perfect if and only if $k = 2^{n-1}(2^n - 1)$ where $2^n - 1$ is a Mersenne prime.
3. **Infinitude of Fermat Pseudoprimes**: Constructive proof that for any integer base $a \ge 2$, there exist infinitely many composite numbers $n$ such that $a^{n-1} \equiv 1 \pmod n$.
4. **Computational Verification & Primality Testing**: Implementation of Miller-Rabin and Fermat primality testing.



### Problem 1: Repunit Sequences & Divisibility
Let $a(n) = 11\dots1$ ($n$ digits). Algebraically:
$$a(n) = \sum_{i=0}^{n-1} 10^i = \frac{10^n - 1}{9}$$
Let $A_p = \{n \in \mathbb{N} : p \mid a(n)\}$.

#### Proof:
1. **Case $p = 2$**: Every repunit number $a(n)$ has units digit 1. Since $1 \equiv 1 \pmod 2$, $a(n)$ is always odd. Hence $2 \nmid a(n)$ for all $n$, so $A_2 = \emptyset$.
2. **Case $p = 5$**: The units digit of $a(n)$ is always 1. A number is divisible by 5 if and only if its last digit is 0 or 5. Hence $5 \nmid a(n)$ for all $n$, so $A_5 = \emptyset$.
3. **Case $p = 3$**: The sum of digits of $a(n)$ is $n$. By the divisibility rule for 3, $3 \mid a(n) \iff 3 \mid n$. Thus $A_3 = \{3k : k \in \mathbb{N}\}$, which is infinite.
4. **General Case ($p > 5$)**:
   $p \mid a(n) \iff 9a(n) \equiv 0 \pmod p \iff 10^n \equiv 1 \pmod p$.
   Since $\gcd(10, p) = 1$, by Fermat's Little Theorem:
   $$10^{p-1} \equiv 1 \pmod p$$
   Let $d = \text{ord}_p(10)$ be the multiplicative order of 10 modulo $p$. Then $10^n \equiv 1 \pmod p \iff d \mid n$.  
   Therefore, $A_p = \{k \cdot \text{ord}_p(10) : k \in \mathbb{Z}^+\}$, which is an **infinite set**.



In [1]:
def repunit(n):
    return (10**n - 1) // 9

def find_first_repunit_multiple(p):
    if p in (2, 5):
        return None
    # For p=3, sum of digits is n, divisible by 3 when n=3
    if p == 3:
        return 3
    # Find order of 10 mod (9*p if p divides 9 else p)
    # Actually p | a(n) <=> 10^n = 1 (mod 9*p) if gcd(p, 9) != 1 else mod p
    # For p != 3: 10^n = 1 (mod p) implies p | (10^n - 1). Since gcd(p, 9) = 1, p | (10^n-1)/9.
    mod = p
    cur = 10 % mod
    n = 1
    while cur != 1:
        cur = (cur * 10) % mod
        n += 1
    return n

primes_to_test = [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
for p in primes_to_test:
    first_n = find_first_repunit_multiple(p)
    if first_n is None:
        print(f"Prime p = {p:2d} -> A_p is EMPTY (No repunit is divisible by {p})")
    else:
        print(f"Prime p = {p:2d} -> Minimal n = {first_n:2d} | a({first_n}) % {p} = {repunit(first_n) % p} | A_{p} = {{{first_n}k : k >= 1}}")



Prime p =  2 -> A_p is EMPTY (No repunit is divisible by 2)


Prime p =  3 -> Minimal n =  3 | a(3) % 3 = 0 | A_3 = {3k : k >= 1}


Prime p =  5 -> A_p is EMPTY (No repunit is divisible by 5)


Prime p =  7 -> Minimal n =  6 | a(6) % 7 = 0 | A_7 = {6k : k >= 1}


Prime p = 11 -> Minimal n =  2 | a(2) % 11 = 0 | A_11 = {2k : k >= 1}


Prime p = 13 -> Minimal n =  6 | a(6) % 13 = 0 | A_13 = {6k : k >= 1}


Prime p = 17 -> Minimal n = 16 | a(16) % 17 = 0 | A_17 = {16k : k >= 1}


Prime p = 19 -> Minimal n = 18 | a(18) % 19 = 0 | A_19 = {18k : k >= 1}


Prime p = 23 -> Minimal n = 22 | a(22) % 23 = 0 | A_23 = {22k : k >= 1}


Prime p = 29 -> Minimal n = 28 | a(28) % 29 = 0 | A_29 = {28k : k >= 1}


### Problem 2: Euclid-Euler Theorem on Even Perfect Numbers
A positive integer $k$ is **perfect** if the sum of its positive divisors $\sigma(k) = 2k$.

#### Statement:
An integer $k$ is an even perfect number if and only if:
$$k = 2^{n-1}(2^n - 1) \quad \text{where } 2^n - 1 \text{ is prime (a Mersenne prime)}$$

#### Proof:
**($\Leftarrow$ Euclid):**  
Let $p = 2^n - 1$ be prime. Then $k = 2^{n-1} p$.  
Since $\gcd(2^{n-1}, p) = 1$ and the divisor sum function $\sigma$ is multiplicative:
$$\sigma(k) = \sigma(2^{n-1}) \cdot \sigma(p) = (2^n - 1) \cdot (p + 1)$$
Substituting $p = 2^n - 1$:
$$\sigma(k) = (2^n - 1) \cdot (2^n) = 2 \cdot [2^{n-1}(2^n - 1)] = 2k$$
Thus, $k$ is perfect.

**($\Rightarrow$ Euler):**  
Let $k$ be an even perfect number. Factor out powers of 2: $k = 2^{n-1} m$ where $m$ is odd and $n \ge 2$.  
Because $k$ is perfect:
$$\sigma(k) = \sigma(2^{n-1})\sigma(m) = (2^n - 1)\sigma(m) = 2k = 2^n m$$
Notice that $(2^n - 1)$ divides $2^n m$. Since $\gcd(2^n - 1, 2^n) = 1$, $(2^n - 1)$ must divide $m$.  
Let $m = (2^n - 1) c$. Then:
$$(2^n - 1)\sigma(m) = 2^n(2^n - 1)c \implies \sigma(m) = 2^n c$$
Now, $m$ has at least two distinct divisors: $m$ and $c = m / (2^n - 1)$ (if $c < m$).  
The sum of these two divisors is $m + c = (2^n - 1)c + c = 2^n c = \sigma(m)$.  
Since the sum of just these two divisors equals the total sum of *all* divisors $\sigma(m)$, $m$ can have no other divisors!  
This implies $c = 1$ and $m$ is prime.  
Hence $m = 2^n - 1$ is prime, completing the proof. $\blacksquare$



In [1]:
def is_prime(n):
    if n < 2: return False
    if n in (2, 3): return True
    if n % 2 == 0 or n % 3 == 0: return False
    i = 5
    while i * i <= n:
        if n % i == 0 or n % (i + 2) == 0: return False
        i += 6
    return True

print("First 7 Even Perfect Numbers (Euclid-Euler Formula):")
count = 0
for n in range(2, 32):
    mersenne = 2**n - 1
    if is_prime(mersenne):
        perfect_num = (2**(n - 1)) * mersenne
        count += 1
        print(f"[{count}] n = {n:2d} | Mersenne Prime M_{n} = 2^{n}-1 = {mersenne:<10d} | Even Perfect Number = {perfect_num}")



First 7 Even Perfect Numbers (Euclid-Euler Formula):


[1] n =  2 | Mersenne Prime M_2 = 2^2-1 = 3          | Even Perfect Number = 6


[2] n =  3 | Mersenne Prime M_3 = 2^3-1 = 7          | Even Perfect Number = 28


[3] n =  5 | Mersenne Prime M_5 = 2^5-1 = 31         | Even Perfect Number = 496


[4] n =  7 | Mersenne Prime M_7 = 2^7-1 = 127        | Even Perfect Number = 8128


[5] n = 13 | Mersenne Prime M_13 = 2^13-1 = 8191       | Even Perfect Number = 33550336


[6] n = 17 | Mersenne Prime M_17 = 2^17-1 = 131071     | Even Perfect Number = 8589869056


[7] n = 19 | Mersenne Prime M_19 = 2^19-1 = 524287     | Even Perfect Number = 137438691328


### Problem 3: Infinitude of Fermat Pseudoprimes to Base $a$
A composite number $n$ is called a **Fermat pseudoprime to base $a$** if:
$$a^{n-1} \equiv 1 \pmod n$$

#### Constructive Proof (Cipolla's Method):
Let $a \ge 2$ be an integer. Let $p$ be any odd prime such that $p \nmid a(a^2 - 1)$.  
Consider the number:
$$n = \frac{a^{2p} - 1}{a^2 - 1} = \sum_{j=0}^{p-1} a^{2j}$$
1. **$n$ is composite**: Since $p$ is an odd prime, we can factor $a^{2p} - 1 = (a^p - 1)(a^p + 1)$. Since $a \ge 2$ and $p \ge 3$, both factors exceed $a^2 - 1$, so $n$ is strictly composite.
2. **$n$ is odd**: If $a$ is even, $n$ is a sum of even powers, each $\equiv 0 \pmod 2$, plus the constant term 1, so $n$ is odd. If $a$ is odd, each $a^{2j}$ is odd, and the sum has $p$ (an odd number of) odd terms, so $n$ is odd.
3. **Divisibility condition**:
   Note that $n - 1 = \frac{a^{2p} - 1}{a^2 - 1} - 1 = \frac{a^{2p} - a^2}{a^2 - 1} = \frac{a^2(a^{2(p-1)} - 1)}{a^2 - 1}$.  
   By Fermat's Little Theorem, $2p \mid (a^2(a^{2(p-1)} - 1))$ because $p \mid (a^{2(p-1)} - 1)$ and $2 \mid (n - 1)$.  
   Therefore, $2p$ divides $n - 1$.  
   Since $a^{2p} \equiv 1 \pmod n$ (as $a^{2p} - 1 = n(a^2 - 1)$), and $2p \mid (n - 1)$:
   $$a^{n-1} = (a^{2p})^{(n-1)/(2p)} \equiv 1^{(n-1)/(2p)} \equiv 1 \pmod n$$
Since there are infinitely many primes $p$, this construction yields **infinitely many Fermat pseudoprimes base $a$**. $\blacksquare$



In [1]:
def verify_fermat_pseudoprime(n, a):
    if is_prime(n): return False # must be composite
    return pow(a, n - 1, n) == 1

print("Small Fermat Pseudoprimes base 2:")
pseudoprimes_base2 = []
candidate = 4
while len(pseudoprimes_base2) < 5:
    if verify_fermat_pseudoprime(candidate, 2):
        pseudoprimes_base2.append(candidate)
    candidate += 1

for psp in pseudoprimes_base2:
    print(f"  n = {psp:<6d} is composite (factors: {[d for d in range(2, int(psp**0.5)+1) if psp%d==0][:2]}) and 2^{psp-1} = 1 (mod {psp})")



Small Fermat Pseudoprimes base 2:


  n = 341    is composite (factors: [11]) and 2^340 = 1 (mod 341)


  n = 561    is composite (factors: [3]) and 2^560 = 1 (mod 561)


  n = 645    is composite (factors: [3]) and 2^644 = 1 (mod 645)


  n = 1105   is composite (factors: [5]) and 2^1104 = 1 (mod 1105)


  n = 1387   is composite (factors: [19]) and 2^1386 = 1 (mod 1387)
